# GenAI Security Analyst

This notebook extends the trained PyTorch security incident triage
classifier with a Generative AI analyst layer.

The predictive model determines the incident triage class.

The GenAI component receives the model prediction, confidence score,
and structured incident context and generates:

- a concise analyst explanation;
- a recommended next action;
- an uncertainty / limitation statement.

The language model does not replace the classifier and does not infer
causal feature importance that the classifier has not provided.

## Imports

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

## Device

In [2]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    "Using device:",
    device
)

Using device: cpu


## Paths

In [3]:
current_path = Path.cwd()

if current_path.name == "notebooks":
    PROJECT_ROOT = current_path.parent
else:
    PROJECT_ROOT = current_path

DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
)

MODELS_DIR = (
    PROJECT_ROOT
    / "models"
)

TRAIN_PATH = (
    DATA_DIR
    / "GUIDE_Train.csv"
)

MODEL_PATH = (
    MODELS_DIR
    / "pytorch_incident_triage.pt"
)

PREPROCESSOR_PATH = (
    MODELS_DIR
    / "pytorch_preprocessing.joblib"
)

print("Model exists:", MODEL_PATH.exists())
print(
    "Preprocessor exists:",
    PREPROCESSOR_PATH.exists()
)

Model exists: True
Preprocessor exists: True


## Load preprocessing artifacts

In [4]:
artifacts = joblib.load(
    PREPROCESSOR_PATH
)

In [5]:
categorical_features = (
    artifacts["categorical_features"]
)

numeric_features = (
    artifacts["numeric_features"]
)

category_maps = (
    artifacts["category_maps"]
)

category_cardinalities = (
    artifacts["category_cardinalities"]
)

embedding_dims = (
    artifacts["embedding_dims"]
)

numeric_medians = (
    artifacts["numeric_medians"]
)

scaler = (
    artifacts["scaler"]
)

label_to_index = (
    artifacts["label_to_index"]
)

index_to_label = (
    artifacts["index_to_label"]
)

UNKNOWN_INDEX = (
    artifacts["unknown_index"]
)

MISSING_TOKEN = (
    artifacts["missing_token"]
)

In [6]:
print(
    "Categorical features:",
    len(categorical_features)
)

print(
    "Numerical features:",
    len(numeric_features)
)

print(
    "Labels:",
    index_to_label
)

Categorical features: 17
Numerical features: 4
Labels: {0: 'BenignPositive', 1: 'FalsePositive', 2: 'TruePositive'}


## Load PyTorch classifier

In [7]:
class IncidentTriageNetwork(nn.Module):

    def __init__(
        self,
        categorical_features,
        category_cardinalities,
        embedding_dims,
        num_numeric_features,
        num_classes=3
    ):
        super().__init__()

        self.categorical_features = categorical_features

        self.embeddings = nn.ModuleList([
            nn.Embedding(
                num_embeddings=category_cardinalities[column],
                embedding_dim=embedding_dims[column]
            )
            for column in categorical_features
        ])

        total_embedding_dim = sum(
            embedding_dims[column]
            for column in categorical_features
        )

        total_input_dim = (
            total_embedding_dim
            + num_numeric_features
        )

        self.network = nn.Sequential(
            nn.Linear(total_input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.30),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.20),

            nn.Linear(64, num_classes)
        )


    def forward(
        self,
        categorical_data,
        numerical_data
    ):

        embedded_features = []

        for i, embedding_layer in enumerate(
            self.embeddings
        ):
            embedded = embedding_layer(
                categorical_data[:, i]
            )

            embedded_features.append(
                embedded
            )

        categorical_embeddings = torch.cat(
            embedded_features,
            dim=1
        )

        combined_features = torch.cat(
            [
                categorical_embeddings,
                numerical_data
            ],
            dim=1
        )

        logits = self.network(
            combined_features
        )

        return logits

In [8]:
classifier = IncidentTriageNetwork(
    categorical_features=
        categorical_features,

    category_cardinalities=
        category_cardinalities,

    embedding_dims=
        embedding_dims,

    num_numeric_features=
        len(numeric_features),

    num_classes=3
)

In [9]:
classifier.load_state_dict(
    torch.load(
        MODEL_PATH,
        map_location=device,
        weights_only=True
    )
)

classifier = classifier.to(device)

classifier.eval()

print(
    "PyTorch classifier loaded."
)

PyTorch classifier loaded.


## Load Hugging Face language model

In [10]:
HF_MODEL_NAME = (
    "google/flan-t5-small"
)

In [11]:
tokenizer = (
    AutoTokenizer
    .from_pretrained(
        HF_MODEL_NAME
    )
)

In [12]:
language_model = (
    AutoModelForSeq2SeqLM
    .from_pretrained(
        HF_MODEL_NAME
    )
)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [13]:
language_model = (
    language_model.to(device)
)

language_model.eval()

print(
    "Loaded:",
    HF_MODEL_NAME
)

Loaded: google/flan-t5-small


## First Hugging Face test

In [14]:
test_prompt = """
Explain briefly why cybersecurity
incident triage is useful to a
security analyst.
"""

In [15]:
inputs = tokenizer(
    test_prompt,
    return_tensors="pt",
    truncation=True
)

In [16]:
inputs = {
    key: value.to(device)
    for key, value
    in inputs.items()
}

In [17]:
with torch.no_grad():

    generated_ids = (
        language_model.generate(
            **inputs,
            max_new_tokens=80
        )
    )

In [18]:
generated_text = (
    tokenizer.decode(
        generated_ids[0],
        skip_special_tokens=True
    )
)

print(generated_text)

cybersecurity incidents are often a threat to the public.


## PyTorch classifier inference

In [19]:
demo_df = pd.read_csv(
    TRAIN_PATH,
    nrows=100_000,
    low_memory=False
)

demo_df = demo_df.dropna(
    subset=["IncidentGrade"]
).copy()

print(
    "Available demo rows:",
    len(demo_df)
)

Available demo rows: 99475


## Preprocessing function

In [20]:
def prepare_incident_for_classifier(
    row
):

    # -------------------------
    # Categorical features
    # -------------------------

    categorical_indices = []

    for column in categorical_features:

        value = row[column]

        if pd.isna(value):
            value = MISSING_TOKEN
        else:
            value = str(value)

        category_index = (
            category_maps[column]
            .get(
                value,
                UNKNOWN_INDEX
            )
        )

        categorical_indices.append(
            category_index
        )

    # -------------------------
    # Numerical feature engineering
    # -------------------------

    timestamp = pd.to_datetime(
        row["Timestamp"],
        errors="coerce",
        utc=True
    )

    if pd.isna(timestamp):

        hour = np.nan
        day_of_week = np.nan
        is_weekend = np.nan

    else:

        hour = timestamp.hour
        day_of_week = timestamp.dayofweek

        is_weekend = int(
            day_of_week >= 5
        )

    has_mitre_technique = int(
        pd.notna(
            row["MitreTechniques"]
        )
    )

    numerical_values = {
        "Hour": hour,
        "DayOfWeek": day_of_week,
        "IsWeekend": is_weekend,
        "HasMitreTechnique":
            has_mitre_technique
    }

    numerical_df = pd.DataFrame(
        [numerical_values],
        columns=numeric_features
    ).astype(float)

    numerical_df = (
        numerical_df
        .fillna(numeric_medians)
    )

    numerical_scaled = (
        scaler
        .transform(numerical_df)
        .astype(np.float32)
    )

    # -------------------------
    # PyTorch tensors
    # -------------------------

    categorical_tensor = torch.tensor(
        [categorical_indices],
        dtype=torch.long,
        device=device
    )

    numerical_tensor = torch.tensor(
        numerical_scaled,
        dtype=torch.float32,
        device=device
    )

    return (
        categorical_tensor,
        numerical_tensor
    )

## Prediction function

In [21]:
def predict_incident(
    row
):

    (
        categorical_tensor,
        numerical_tensor
    ) = prepare_incident_for_classifier(
        row
    )

    classifier.eval()

    with torch.no_grad():

        logits = classifier(
            categorical_tensor,
            numerical_tensor
        )

        probabilities = torch.softmax(
            logits,
            dim=1
        )

    predicted_index = int(
        torch.argmax(
            probabilities,
            dim=1
        ).item()
    )

    confidence = float(
        probabilities[
            0,
            predicted_index
        ].item()
    )

    class_probabilities = {
        index_to_label[index]:
            float(
                probabilities[
                    0,
                    index
                ].item()
            )

        for index in range(
            len(index_to_label)
        )
    }

    return {
        "predicted_class":
            index_to_label[
                predicted_index
            ],

        "confidence":
            confidence,

        "probabilities":
            class_probabilities
    }

## Demo incident row

In [22]:
demo_row = (
    demo_df
    .iloc[0]
)

prediction = predict_incident(
    demo_row
)

print(
    "Actual label:",
    demo_row["IncidentGrade"]
)

print(
    "Predicted class:",
    prediction["predicted_class"]
)

print(
    "Confidence:",
    round(
        prediction["confidence"],
        4
    )
)

print(
    "Class probabilities:"
)

for (
    class_name,
    probability
) in prediction["probabilities"].items():

    print(
        f"{class_name:15s}: "
        f"{probability:.4f}"
    )

Actual label: TruePositive
Predicted class: TruePositive
Confidence: 0.9901
Class probabilities:
BenignPositive : 0.0015
FalsePositive  : 0.0085
TruePositive   : 0.9901


## Structured incident context

In [23]:
def format_context_value(
    value
):

    if pd.isna(value):
        return "Not available"

    return str(value)

In [24]:
def build_incident_context(
    row
):

    return {
        "Timestamp":
            format_context_value(
                row["Timestamp"]
            ),

        "Category":
            format_context_value(
                row["Category"]
            ),

        "EntityType":
            format_context_value(
                row["EntityType"]
            ),

        "EvidenceRole":
            format_context_value(
                row["EvidenceRole"]
            ),

        "MitreTechniques":
            format_context_value(
                row["MitreTechniques"]
            )
    }

In [25]:
incident_context = (
    build_incident_context(
        demo_row
    )
)

incident_context

{'Timestamp': '2024-06-04T06:05:15.000Z',
 'Category': 'InitialAccess',
 'EntityType': 'Ip',
 'EvidenceRole': 'Related',
 'MitreTechniques': 'Not available'}

## Structured prompt

In [26]:
def build_analyst_prompt(
    incident_context,
    prediction
):

    context_text = "\n".join(
        f"{key}: {value}"
        for key, value
        in incident_context.items()
    )

    probabilities_text = "\n".join(
        (
            f"{class_name}: "
            f"{probability:.3f}"
        )
        for (
            class_name,
            probability
        )
        in prediction[
            "probabilities"
        ].items()
    )

    prompt = f"""
You are assisting a cybersecurity analyst.

A separate machine learning classifier has already
performed incident triage.

Do not change the classifier prediction.
Do not invent evidence.
Do not claim that a feature caused the prediction.
Use only the information supplied below.

MODEL PREDICTION:
Predicted class:
{prediction["predicted_class"]}

Confidence:
{prediction["confidence"]:.3f}

CLASS PROBABILITIES:
{probabilities_text}

AVAILABLE INCIDENT CONTEXT:
{context_text}

TASK:
Provide a concise operational explanation of the
classifier result and suggest an appropriate next
step for a human security analyst.

Return exactly these three sections:

Assessment:
Recommended action:
Caveat:
"""

    return prompt.strip()

In [27]:
analyst_prompt = build_analyst_prompt(
    incident_context,
    prediction
)

print(analyst_prompt)

You are assisting a cybersecurity analyst.

A separate machine learning classifier has already
performed incident triage.

Do not change the classifier prediction.
Do not invent evidence.
Do not claim that a feature caused the prediction.
Use only the information supplied below.

MODEL PREDICTION:
Predicted class:
TruePositive

Confidence:
0.990

CLASS PROBABILITIES:
BenignPositive: 0.001
FalsePositive: 0.008
TruePositive: 0.990

AVAILABLE INCIDENT CONTEXT:
Timestamp: 2024-06-04T06:05:15.000Z
Category: InitialAccess
EntityType: Ip
EvidenceRole: Related
MitreTechniques: Not available

TASK:
Provide a concise operational explanation of the
classifier result and suggest an appropriate next
step for a human security analyst.

Return exactly these three sections:

Assessment:
Recommended action:
Caveat:


## Generation function

In [28]:
def generate_analyst_response(
    prompt
):

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    )

    inputs = {
        key: value.to(device)
        for key, value
        in inputs.items()
    }

    with torch.no_grad():

        generated_ids = (
            language_model.generate(
                **inputs,
                max_new_tokens=128,
                do_sample=False,
                num_beams=4
            )
        )

    response = tokenizer.decode(
        generated_ids[0],
        skip_special_tokens=True
    )

    return response

In [29]:
analyst_response = (
    generate_analyst_response(
        analyst_prompt
    )
)

print(
    analyst_response
)

TASK


## Simplified prompt experiment

In [30]:
def build_simple_analyst_prompt(
    incident_context,
    prediction
):

    return f"""
Cybersecurity incident triage result:

Prediction: {prediction["predicted_class"]}
Confidence: {prediction["confidence"]:.3f}
Category: {incident_context["Category"]}
Entity type: {incident_context["EntityType"]}
Evidence role: {incident_context["EvidenceRole"]}
MITRE techniques: {incident_context["MitreTechniques"]}

Using only this information, respond in three short lines:

Assessment: explain the prediction.
Recommended action: suggest what the analyst should check next.
Caveat: state what information is missing.

Do not invent facts.
""".strip()

In [31]:
simple_prompt = build_simple_analyst_prompt(
    incident_context,
    prediction
)

print(simple_prompt)

Cybersecurity incident triage result:

Prediction: TruePositive
Confidence: 0.990
Category: InitialAccess
Entity type: Ip
Evidence role: Related
MITRE techniques: Not available

Using only this information, respond in three short lines:

Assessment: explain the prediction.
Recommended action: suggest what the analyst should check next.
Caveat: state what information is missing.

Do not invent facts.


In [32]:
simple_response = generate_analyst_response(
    simple_prompt
)

print(simple_response)

Using only this information, respond in three short lines: Assessment: Explain the prediction. Recommended action: suggest what the analyst should check.


## Language model capability comparison

In [33]:
HF_BASE_MODEL_NAME = "google/flan-t5-base"

In [34]:
base_tokenizer = (
    AutoTokenizer
    .from_pretrained(
        HF_BASE_MODEL_NAME
    )
)

In [35]:
base_language_model = (
    AutoModelForSeq2SeqLM
    .from_pretrained(
        HF_BASE_MODEL_NAME
    )
)

base_language_model = (
    base_language_model.to(device)
)

base_language_model.eval()

print(
    "Loaded:",
    HF_BASE_MODEL_NAME
)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Loaded: google/flan-t5-base


In [36]:
def generate_response(
    prompt,
    tokenizer,
    model
):

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    )

    inputs = {
        key: value.to(device)
        for key, value
        in inputs.items()
    }

    with torch.no_grad():

        generated_ids = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            num_beams=4
        )

    response = tokenizer.decode(
        generated_ids[0],
        skip_special_tokens=True
    )

    return response

In [37]:
small_response = generate_response(
    simple_prompt,
    tokenizer,
    language_model
)

In [38]:
base_response = generate_response(
    simple_prompt,
    base_tokenizer,
    base_language_model
)

In [39]:
print("FLAN-T5 Small")
print("-" * 40)
print(small_response)

print()

print("FLAN-T5 Base")
print("-" * 40)
print(base_response)

FLAN-T5 Small
----------------------------------------
Using only this information, respond in three short lines: Assessment: Explain the prediction. Recommended action: suggest what the analyst should check.

FLAN-T5 Base
----------------------------------------
This is a summary of the information that we received about the incident.


### FLAN-T5 experiment conclusion

Both FLAN-T5-small and FLAN-T5-base failed to reliably follow the
structured analyst instruction for this incident.

Simplifying the prompt did not resolve the issue. The larger base model
produced a more fluent response, but still failed to provide the requested
Assessment, Recommended action, and Caveat fields.

The next experiment evaluates an instruction-tuned causal language model
while keeping the same incident context and task.

In [40]:
import gc

del language_model
del base_language_model

gc.collect()

print(
    "FLAN-T5 models released from memory."
)

FLAN-T5 models released from memory.


## Instruction-tuned causal language model

In [41]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

In [42]:
SMOL_MODEL_NAME = (
    "HuggingFaceTB/SmolLM2-360M-Instruct"
)

In [43]:
smol_tokenizer = (
    AutoTokenizer
    .from_pretrained(
        SMOL_MODEL_NAME
    )
)

In [44]:
smol_model = (
    AutoModelForCausalLM
    .from_pretrained(
        SMOL_MODEL_NAME
    )
)

smol_model = smol_model.to(
    device
)

smol_model.eval()

print(
    "Loaded:",
    SMOL_MODEL_NAME
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded: HuggingFaceTB/SmolLM2-360M-Instruct


In [45]:
messages = [
    {
        "role": "user",
        "content": simple_prompt
    }
]

In [46]:
chat_text = (
    smol_tokenizer
    .apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
)

print(
    chat_text
)

<|im_start|>system
You are a helpful AI assistant named SmolLM, trained by Hugging Face<|im_end|>
<|im_start|>user
Cybersecurity incident triage result:

Prediction: TruePositive
Confidence: 0.990
Category: InitialAccess
Entity type: Ip
Evidence role: Related
MITRE techniques: Not available

Using only this information, respond in three short lines:

Assessment: explain the prediction.
Recommended action: suggest what the analyst should check next.
Caveat: state what information is missing.

Do not invent facts.<|im_end|>
<|im_start|>assistant



In [47]:
smol_inputs = (
    smol_tokenizer
    .apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    )
)

In [48]:
smol_inputs = {
    key: value.to(device)
    for key, value
    in smol_inputs.items()
}

In [49]:
print(
    "Input shape:",
    smol_inputs[
        "input_ids"
    ].shape
)

Input shape: torch.Size([1, 128])


In [50]:
with torch.no_grad():

    smol_outputs = (
        smol_model.generate(
            **smol_inputs,
            max_new_tokens=160,
            do_sample=False
        )
    )

In [51]:
input_length = (
    smol_inputs[
        "input_ids"
    ].shape[1]
)

In [52]:
generated_tokens = (
    smol_outputs[
        0,
        input_length:
    ]
)

In [53]:
smol_response = (
    smol_tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )
)

print(
    smol_response
)

Prediction: TruePositive
Confidence: 0.990
Category: InitialAccess
Entity type: Ip
Evidence role: Related
MITRE techniques: Not available

The analyst should check for any potential vulnerabilities in the network, such as firewall rules or access controls, to ensure the incident is not related to a known vulnerability.


In [54]:
print(
    "FLAN-T5 Small"
)
print("-" * 50)
print(small_response)

print()

print(
    "FLAN-T5 Base"
)
print("-" * 50)
print(base_response)

print()

print(
    "SmolLM2-360M-Instruct"
)
print("-" * 50)
print(smol_response)

FLAN-T5 Small
--------------------------------------------------
Using only this information, respond in three short lines: Assessment: Explain the prediction. Recommended action: suggest what the analyst should check.

FLAN-T5 Base
--------------------------------------------------
This is a summary of the information that we received about the incident.

SmolLM2-360M-Instruct
--------------------------------------------------
Prediction: TruePositive
Confidence: 0.990
Category: InitialAccess
Entity type: Ip
Evidence role: Related
MITRE techniques: Not available

The analyst should check for any potential vulnerabilities in the network, such as firewall rules or access controls, to ensure the incident is not related to a known vulnerability.


## Role-based structured prompting

In [55]:
system_message = """
You are a cybersecurity incident triage assistant.

A separate machine learning classifier has already made the prediction.
Your role is only to help a human analyst interpret the supplied result.

Rules:
- Use only the supplied information.
- Do not change the classifier prediction.
- Do not claim that any feature caused the prediction.
- Do not invent indicators, vulnerabilities, attacks, or evidence.
- If information is unavailable, explicitly say so.
- Keep the response concise.

Return exactly three lines using these prefixes:
Assessment:
Recommended action:
Caveat:
""".strip()

In [56]:
def build_smol_user_message(
    incident_context,
    prediction
):

    return f"""
Classifier prediction: {prediction["predicted_class"]}
Confidence: {prediction["confidence"]:.3f}

Incident context:
Category: {incident_context["Category"]}
Entity type: {incident_context["EntityType"]}
Evidence role: {incident_context["EvidenceRole"]}
MITRE techniques: {incident_context["MitreTechniques"]}

Explain this result for a human security analyst.
""".strip()

In [57]:
smol_user_message = build_smol_user_message(
    incident_context,
    prediction
)

print(smol_user_message)

Classifier prediction: TruePositive
Confidence: 0.990

Incident context:
Category: InitialAccess
Entity type: Ip
Evidence role: Related
MITRE techniques: Not available

Explain this result for a human security analyst.


In [58]:
structured_messages = [
    {
        "role": "system",
        "content": system_message
    },
    {
        "role": "user",
        "content": smol_user_message
    }
]

In [59]:
structured_chat_text = (
    smol_tokenizer
    .apply_chat_template(
        structured_messages,
        tokenize=False,
        add_generation_prompt=True
    )
)

print(structured_chat_text)

<|im_start|>system
You are a cybersecurity incident triage assistant.

A separate machine learning classifier has already made the prediction.
Your role is only to help a human analyst interpret the supplied result.

Rules:
- Use only the supplied information.
- Do not change the classifier prediction.
- Do not claim that any feature caused the prediction.
- Do not invent indicators, vulnerabilities, attacks, or evidence.
- If information is unavailable, explicitly say so.
- Keep the response concise.

Return exactly three lines using these prefixes:
Assessment:
Recommended action:
Caveat:<|im_end|>
<|im_start|>user
Classifier prediction: TruePositive
Confidence: 0.990

Incident context:
Category: InitialAccess
Entity type: Ip
Evidence role: Related
MITRE techniques: Not available

Explain this result for a human security analyst.<|im_end|>
<|im_start|>assistant



In [60]:
def generate_chat_response(
    messages,
    tokenizer,
    model,
    device,
    max_new_tokens=160
):

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(device)
        for key, value
        in inputs.items()
    }

    input_length = (
        inputs["input_ids"]
        .shape[1]
    )

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    generated_tokens = outputs[
        0,
        input_length:
    ]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response.strip()

In [61]:
structured_smol_response = (
    generate_chat_response(
        structured_messages,
        smol_tokenizer,
        smol_model,
        device
    )
)

print(
    structured_smol_response
)

Classifier prediction: TruePositive
Confidence: 0.990

This result indicates that the classifier has correctly identified the entity as an IP address, with a confidence level of 99%. The presence of the "Ip" entity type in the evidence role suggests that the classifier is able to distinguish between IP addresses and other types of network identifiers. The "Not available" caveat indicates that the classifier is not able to provide information on the specific MITRE techniques used to identify the IP address.


## Response validation

In [62]:
REQUIRED_SECTIONS = [
    "Assessment:",
    "Recommended action:",
    "Caveat:"
]

In [63]:
def validate_analyst_response(
    response
):

    missing_sections = [
        section
        for section in REQUIRED_SECTIONS
        if section not in response
    ]

    is_valid = (
        len(missing_sections) == 0
    )

    return {
        "is_valid": is_valid,
        "missing_sections": missing_sections
    }

In [64]:
validation_result = (
    validate_analyst_response(
        structured_smol_response
    )
)

print(
    validation_result
)

{'is_valid': False, 'missing_sections': ['Assessment:', 'Recommended action:', 'Caveat:']}


## Larger instruction model experiment

In [65]:
import gc

if "smol_model" in globals():
    del smol_model

gc.collect()

print("Previous model memory cleared.")

Previous model memory cleared.


In [ ]:
SMOL_LARGE_MODEL_NAME = (
    "HuggingFaceTB/SmolLM2-1.7B-Instruct"
)

large_tokenizer = (
    AutoTokenizer
    .from_pretrained(
        SMOL_LARGE_MODEL_NAME
    )
)

: 

In [ ]:
large_model = (
    AutoModelForCausalLM
    .from_pretrained(
        SMOL_LARGE_MODEL_NAME
    )
)

large_model = large_model.to(device)
large_model.eval()

print(
    "Loaded:",
    SMOL_LARGE_MODEL_NAME
)

In [ ]:
large_smol_response = (
    generate_chat_response(
        structured_messages,
        large_tokenizer,
        large_model,
        device,
        max_new_tokens=160
    )
)

print(
    large_smol_response
)

In [ ]:
large_validation_result = (
    validate_analyst_response(
        large_smol_response
    )
)

print(large_validation_result)

## Isolated SmolLM2-1.7B experiment

In [1]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

Device: cpu


In [2]:
SMOL_LARGE_MODEL_NAME = (
    "HuggingFaceTB/SmolLM2-1.7B-Instruct"
)

In [5]:
system_message_large_test = """
You are a cybersecurity incident triage assistant.

A separate machine learning classifier has already made the prediction.
Your role is only to help a human analyst interpret the supplied result.

Rules:
- Use only the supplied information.
- Do not change the classifier prediction.
- Do not claim that any feature caused the prediction.
- Do not invent indicators, vulnerabilities, attacks, or evidence.
- If information is unavailable, explicitly say so.
- Keep the response concise.

Return exactly three lines using these prefixes:
Assessment:
Recommended action:
Caveat:
""".strip()

In [6]:
user_message_large_test = """
Classifier prediction: TruePositive
Confidence: 0.990

Incident context:
Category: InitialAccess
Entity type: Ip
Evidence role: Related
MITRE techniques: Not available

Explain this result for a human security analyst.
""".strip()

In [7]:
large_test_messages = [
    {
        "role": "system",
        "content": system_message_large_test
    },
    {
        "role": "user",
        "content": user_message_large_test
    }
]

In [3]:
large_tokenizer = (
    AutoTokenizer
    .from_pretrained(
        SMOL_LARGE_MODEL_NAME
    )
)

print("Tokenizer loaded successfully.")

Tokenizer loaded successfully.


In [4]:
large_model = (
    AutoModelForCausalLM
    .from_pretrained(
        SMOL_LARGE_MODEL_NAME,
        dtype=torch.float16,
        low_cpu_mem_usage=True
    )
)

large_model.eval()

print(
    "Model loaded successfully:",
    SMOL_LARGE_MODEL_NAME
)

print(
    "Model dtype:",
    next(large_model.parameters()).dtype
)

: 

### Local model feasibility conclusion

SmolLM2-1.7B-Instruct could not be loaded reliably on the available
CPU environment, even when using float16 weights and
low-memory loading.

The local experiments therefore demonstrated a practical trade-off
between language-model capability and hardware requirements.

The GenAI layer is moved to hosted inference while the predictive
PyTorch classifier remains local.

## Hosted Hugging Face inference

In [20]:
from pathlib import Path
import os

from dotenv import load_dotenv
from huggingface_hub import InferenceClient


current_path = Path.cwd()

if current_path.name == "notebooks":
    PROJECT_ROOT = current_path.parent
else:
    PROJECT_ROOT = current_path


ENV_PATH = (
    PROJECT_ROOT
    / ".env"
)


print(
    ".env path:",
    ENV_PATH
)

print(
    ".env exists:",
    ENV_PATH.exists()
)


load_dotenv(
    ENV_PATH,
    override=True
)


HF_TOKEN = os.getenv(
    "HF_TOKEN"
)


print(
    "HF token available:",
    bool(HF_TOKEN)
)

.env path: c:\Users\zaaaa\OneDrive\Υπολογιστής\New folder\Python Projects\ai-security-incident-triage\.env
.env exists: True
HF token available: True


In [21]:
if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN was not found. "
        "Check the project .env file."
    )

In [22]:
hf_client = InferenceClient(
    provider="auto",
    api_key=HF_TOKEN
)

In [23]:
HOSTED_MODEL_NAME = (
    "Qwen/Qwen3-8B"
)

In [24]:
system_message_large_test = """
You are a cybersecurity incident triage assistant.

A separate machine learning classifier has already made the prediction.
Your role is only to help a human analyst interpret the supplied result.

Rules:
- Use only the supplied information.
- Do not change the classifier prediction.
- Do not claim that any feature caused the prediction.
- Do not invent indicators, vulnerabilities, attacks, or evidence.
- If information is unavailable, explicitly say so.
- Keep the response concise.

Return exactly three lines using these prefixes:
Assessment:
Recommended action:
Caveat:
""".strip()

In [25]:
user_message_large_test = """
Classifier prediction: TruePositive
Confidence: 0.990

Incident context:
Category: InitialAccess
Entity type: Ip
Evidence role: Related
MITRE techniques: Not available

Explain this result for a human security analyst.

/no_think
""".strip()

In [26]:
hosted_messages = [
    {
        "role": "system",
        "content":
            system_message_large_test
    },
    {
        "role": "user",
        "content":
            user_message_large_test
    }
]

In [27]:
completion = (
    hf_client
    .chat
    .completions
    .create(
        model=HOSTED_MODEL_NAME,
        messages=hosted_messages,
        max_tokens=256,
        temperature=0.7,
        top_p=0.8
    )
)

In [28]:
print(completion)

ChatCompletionOutput(choices=[ChatCompletionOutputComplete(finish_reason='stop', index=0, message=ChatCompletionOutputMessage(role='assistant', content='\n\nAssessment: The classifier identifies this as a true positive, indicating a high likelihood of a malicious initial access event originating from the IP address.  \nRecommended action: Investigate the IP address for potential compromise, monitor related network activity, and consider blocking or isolating the source.  \nCaveat: MITRE techniques are not available, limiting the ability to map this event to specific attack patterns.', reasoning=None, tool_call_id=None, tool_calls=[], reasoning_content='\n\n'), logprobs=None, stop_reason=None)], created=1788784784, id='chatcmpl-e005fd8f-1596-4e65-97fc-8cf7f13411e3', model='Qwen/Qwen3-8B', system_fingerprint='vllm-0.21.0-9aa9c829', usage=ChatCompletionOutputUsage(completion_tokens=84, prompt_tokens=177, total_tokens=261, prompt_tokens_details=None), object='chat.completion', prompt_logpr

In [29]:
message = completion.choices[0].message

print(
    "Finish reason:",
    completion.choices[0].finish_reason
)

print(
    "Content:",
    message.content
)

print(
    "Reasoning:",
    getattr(
        message,
        "reasoning_content",
        None
    )
)

print(
    "Usage:",
    completion.usage
)

Finish reason: stop
Content: 

Assessment: The classifier identifies this as a true positive, indicating a high likelihood of a malicious initial access event originating from the IP address.  
Recommended action: Investigate the IP address for potential compromise, monitor related network activity, and consider blocking or isolating the source.  
Caveat: MITRE techniques are not available, limiting the ability to map this event to specific attack patterns.
Reasoning: 


Usage: ChatCompletionOutputUsage(completion_tokens=84, prompt_tokens=177, total_tokens=261, prompt_tokens_details=None)


In [30]:
hosted_response = (
    completion
    .choices[0]
    .message
    .content
)

print(hosted_response)



Assessment: The classifier identifies this as a true positive, indicating a high likelihood of a malicious initial access event originating from the IP address.  
Recommended action: Investigate the IP address for potential compromise, monitor related network activity, and consider blocking or isolating the source.  
Caveat: MITRE techniques are not available, limiting the ability to map this event to specific attack patterns.


In [32]:
REQUIRED_SECTIONS = [
    "Assessment:",
    "Recommended action:",
    "Caveat:"
]

In [33]:
def validate_analyst_response(
    response
):

    if not isinstance(
        response,
        str
    ):

        return {
            "is_valid": False,
            "missing_sections":
                REQUIRED_SECTIONS.copy(),
            "reason":
                "Response is not text."
        }

    missing_sections = [
        section
        for section
        in REQUIRED_SECTIONS
        if section not in response
    ]

    return {
        "is_valid":
            len(
                missing_sections
            ) == 0,

        "missing_sections":
            missing_sections,

        "reason":
            (
                "Valid response."
                if not missing_sections
                else
                "Required sections missing."
            )
    }

In [34]:
hosted_validation_result = (
    validate_analyst_response(
        hosted_response
    )
)

print(
    hosted_validation_result
)

{'is_valid': True, 'missing_sections': [], 'reason': 'Valid response.'}


## Grounded GenAI response

In [57]:
grounded_system_message = """
You are a cybersecurity incident triage assistant.

A separate machine learning classifier has already produced
the incident triage prediction.

Your task is to summarize the supplied result for a human analyst.

GROUNDING RULES:
- Use only facts explicitly supplied in the user message.
- Do not infer that an IP address, device, account, or other entity is malicious.
- Do not infer that an entity is the source, destination, attacker, or victim.
- Do not state that a specific attack occurred unless it is explicitly supplied.
- Do not claim that any context field caused the classifier prediction.
- The incident context shown to you is explanation context only.
  Do not claim that these fields caused, determined, or were the complete
  inputs used for the classifier prediction.
- Do not recommend blocking, isolating, disabling, deleting, or otherwise
  taking containment action when malicious behavior has not been explicitly confirmed.
- Missing information must remain unknown.
- The classifier confidence is model confidence, not proof that the prediction is correct.

Return exactly three lines:

Assessment:
Recommended action:
Caveat:
""".strip()

In [58]:
grounded_user_message = """
Classifier prediction: TruePositive
Confidence: 0.990

Incident context:
Category: InitialAccess
Entity type: Ip
Evidence role: Related
MITRE techniques: Not available

Explain the supplied classifier result conservatively for a human analyst.

/no_think
""".strip()

In [59]:
grounded_messages = [
    {
        "role": "system",
        "content": grounded_system_message
    },
    {
        "role": "user",
        "content": grounded_user_message
    }
]

In [60]:
grounded_completion = (
    hf_client
    .chat
    .completions
    .create(
        model=HOSTED_MODEL_NAME,
        messages=grounded_messages,
        max_tokens=256,
        temperature=0.7,
        top_p=0.8
    )
)

In [61]:
grounded_response = (
    grounded_completion
    .choices[0]
    .message
    .content
)

print(grounded_response)



Assessment: The classifier identified a potential true positive incident related to initial access with an IP entity.  
Recommended action: Review the IP entity for suspicious activity and investigate further without taking containment actions.  
Caveat: The classifier's prediction is based on the provided context, and no explicit malicious behavior has been confirmed.


In [62]:
grounded_validation_result = (
    validate_analyst_response(
        grounded_response
    )
)

print(
    grounded_validation_result
)

{'is_valid': True, 'missing_sections': [], 'reason': 'Valid response.'}


In [63]:
FORBIDDEN_PHRASES = [
    "malicious ip",
    "source ip",
    "originating from the ip",
    "block the ip",
    "block the source",
    "isolate the source",
    "confirmed attack",
    "prediction is based on the provided context",
    "prediction is based on the supplied context"
]

In [64]:
def validate_grounding(
    response
):

    if not isinstance(response, str):

        return {
            "is_grounded": False,
            "violations": [
                "Response is not text."
            ]
        }

    normalized_response = (
        response.lower()
    )

    violations = [
        phrase
        for phrase in FORBIDDEN_PHRASES
        if phrase in normalized_response
    ]

    return {
        "is_grounded":
            len(violations) == 0,

        "violations":
            violations
    }

In [65]:
grounding_result = (
    validate_grounding(
        grounded_response
    )
)

print(
    grounding_result
)

{'is_grounded': False, 'violations': ['prediction is based on the provided context']}


## Safe fallback and confidence routing

In [66]:
def build_safe_fallback(
    incident_context,
    prediction
):

    mitre_value = (
        incident_context[
            "MitreTechniques"
        ]
    )

    if (
        mitre_value
        == "Not available"
    ):
        mitre_text = (
            "No MITRE technique information "
            "was supplied."
        )
    else:
        mitre_text = (
            "MITRE technique information "
            "is available in the incident context."
        )

    return (
        f"Assessment: The classifier predicted "
        f"{prediction['predicted_class']} with a model "
        f"confidence score of {prediction['confidence']:.3f}. "
        f"The supplied explanation context includes category "
        f"{incident_context['Category']}, entity type "
        f"{incident_context['EntityType']}, and evidence role "
        f"{incident_context['EvidenceRole']}.\n"
        f"Recommended action: Review the available incident "
        f"evidence and associated context before making a final "
        f"analyst disposition.\n"
        f"Caveat: {mitre_text} The explanation context does not "
        f"establish why the classifier produced its prediction."
    )

### Routing logic test using frozen classifier output

The following values are a frozen snapshot of the classifier prediction
and incident context produced earlier in this notebook. They are reused
only to test the routing and fallback logic after the kernel restart.

In [71]:
routing_demo_prediction = {
    "predicted_class": "TruePositive",
    "confidence": 0.9901,
    "probabilities": {
        "BenignPositive": 0.0015,
        "FalsePositive": 0.0085,
        "TruePositive": 0.9901
    }
}


routing_demo_context = {
    "Timestamp": "2024-06-04T06:05:15.000Z",
    "Category": "InitialAccess",
    "EntityType": "Ip",
    "EvidenceRole": "Related",
    "MitreTechniques": "Not available"
}

In [73]:
safe_fallback_response = (
    build_safe_fallback(
        routing_demo_context,
        routing_demo_prediction
    )
)

print(
    safe_fallback_response
)

Assessment: The classifier predicted TruePositive with a model confidence score of 0.990. The supplied explanation context includes category InitialAccess, entity type Ip, and evidence role Related.
Recommended action: Review the available incident evidence and associated context before making a final analyst disposition.
Caveat: No MITRE technique information was supplied. The explanation context does not establish why the classifier produced its prediction.


In [82]:
GENAI_ROUTING_THRESHOLD = 0.80

In [83]:
def decide_analyst_response(
    prediction,
    incident_context,
    llm_response,
    structure_result,
    grounding_result,
    confidence_threshold=GENAI_ROUTING_THRESHOLD
):

    confidence = (
        prediction[
            "confidence"
        ]
    )

    if confidence < confidence_threshold:

        return {
            "route": "safe_fallback",
            "reason": "low_classifier_confidence",
            "response": build_safe_fallback(
                incident_context,
                prediction
            )
        }

    if not structure_result[
        "is_valid"
    ]:

        return {
            "route": "safe_fallback",
            "reason": "invalid_llm_structure",
            "response": build_safe_fallback(
                incident_context,
                prediction
            )
        }

    if not grounding_result[
        "is_grounded"
    ]:

        return {
            "route": "safe_fallback",
            "reason": "grounding_violation",
            "response": build_safe_fallback(
                incident_context,
                prediction
            )
        }

    return {
        "route": "llm_response",
        "reason": "validated_llm_response",
        "response": llm_response
    }

In [84]:
final_decision = (
    decide_analyst_response(
        prediction=routing_demo_prediction,
        incident_context=routing_demo_context,
        llm_response=grounded_response,
        structure_result=grounded_validation_result,
        grounding_result=grounding_result
    )
)

print(
    "Route:",
    final_decision["route"]
)

print(
    "Reason:",
    final_decision["reason"]
)

print()

print(
    final_decision["response"]
)

Route: safe_fallback
Reason: grounding_violation

Assessment: The classifier predicted TruePositive with a model confidence score of 0.990. The supplied explanation context includes category InitialAccess, entity type Ip, and evidence role Related.
Recommended action: Review the available incident evidence and associated context before making a final analyst disposition.
Caveat: No MITRE technique information was supplied. The explanation context does not establish why the classifier produced its prediction.


## Reusable incident analysis pipeline

In [86]:
def build_grounded_messages(
    prediction,
    incident_context
):

    system_message = """
You are a cybersecurity incident triage assistant.

A separate machine learning classifier has already produced
the incident triage prediction.

Your task is to summarize the supplied result for a human analyst.

GROUNDING RULES:
- Use only facts explicitly supplied in the user message.
- Do not infer that an IP address, device, account, or other entity is malicious.
- Do not infer that an entity is the source, destination, attacker, or victim.
- Do not state that a specific attack occurred unless it is explicitly supplied.
- Do not claim that any context field caused the classifier prediction.
- The incident context shown to you is explanation context only.
  Do not claim that these fields caused, determined, or were the complete
  inputs used for the classifier prediction.
- Do not recommend blocking, isolating, disabling, deleting, or otherwise
  taking containment action when malicious behavior has not been explicitly confirmed.
- Missing information must remain unknown.
- The classifier confidence is model confidence, not proof that the prediction is correct.

Return exactly three lines:

Assessment:
Recommended action:
Caveat:
""".strip()

    user_message = f"""
Classifier prediction: {prediction["predicted_class"]}
Confidence: {prediction["confidence"]:.3f}

Incident context:
Category: {incident_context["Category"]}
Entity type: {incident_context["EntityType"]}
Evidence role: {incident_context["EvidenceRole"]}
MITRE techniques: {incident_context["MitreTechniques"]}

Explain the supplied classifier result conservatively for a human analyst.

/no_think
""".strip()

    return [
        {
            "role": "system",
            "content": system_message
        },
        {
            "role": "user",
            "content": user_message
        }
    ]

In [87]:
def generate_hosted_analyst_response(
    client,
    model_name,
    prediction,
    incident_context
):

    messages = build_grounded_messages(
        prediction,
        incident_context
    )

    completion = (
        client
        .chat
        .completions
        .create(
            model=model_name,
            messages=messages,
            max_tokens=256,
            temperature=0.7,
            top_p=0.8
        )
    )

    return (
        completion
        .choices[0]
        .message
        .content
    )

In [88]:
def analyze_incident(
    prediction,
    incident_context,
    client,
    model_name,
    confidence_threshold=GENAI_ROUTING_THRESHOLD
):

    confidence = (
        prediction[
            "confidence"
        ]
    )

    # Operational routing threshold.
    # This is NOT a calibrated probability.
    if confidence < confidence_threshold:

        return {
            "route": "safe_fallback",
            "reason": "low_classifier_confidence",
            "response": build_safe_fallback(
                incident_context,
                prediction
            )
        }

    try:

        llm_response = (
            generate_hosted_analyst_response(
                client=client,
                model_name=model_name,
                prediction=prediction,
                incident_context=incident_context
            )
        )

    except Exception as exc:

        return {
            "route": "safe_fallback",
            "reason": "llm_inference_error",
            "response": build_safe_fallback(
                incident_context,
                prediction
            ),
            "error_type": type(exc).__name__
        }

    structure_result = (
        validate_analyst_response(
            llm_response
        )
    )

    grounding_result = (
        validate_grounding(
            llm_response
        )
    )

    return decide_analyst_response(
        prediction=prediction,
        incident_context=incident_context,
        llm_response=llm_response,
        structure_result=structure_result,
        grounding_result=grounding_result,
        confidence_threshold=confidence_threshold
    )

In [89]:
analysis_result = (
    analyze_incident(
        prediction=routing_demo_prediction,
        incident_context=routing_demo_context,
        client=hf_client,
        model_name=HOSTED_MODEL_NAME
    )
)

print(
    "Route:",
    analysis_result["route"]
)

print(
    "Reason:",
    analysis_result["reason"]
)

print()

print(
    analysis_result["response"]
)

Route: safe_fallback
Reason: grounding_violation

Assessment: The classifier predicted TruePositive with a model confidence score of 0.990. The supplied explanation context includes category InitialAccess, entity type Ip, and evidence role Related.
Recommended action: Review the available incident evidence and associated context before making a final analyst disposition.
Caveat: No MITRE technique information was supplied. The explanation context does not establish why the classifier produced its prediction.


In [90]:
low_confidence_prediction = {
    **routing_demo_prediction,
    "confidence": 0.55
}

In [91]:
low_confidence_result = (
    analyze_incident(
        prediction=low_confidence_prediction,
        incident_context=routing_demo_context,
        client=hf_client,
        model_name=HOSTED_MODEL_NAME
    )
)

print(
    "Route:",
    low_confidence_result["route"]
)

print(
    "Reason:",
    low_confidence_result["reason"]
)

print()

print(
    low_confidence_result["response"]
)

Route: safe_fallback
Reason: low_classifier_confidence

Assessment: The classifier predicted TruePositive with a model confidence score of 0.550. The supplied explanation context includes category InitialAccess, entity type Ip, and evidence role Related.
Recommended action: Review the available incident evidence and associated context before making a final analyst disposition.
Caveat: No MITRE technique information was supplied. The explanation context does not establish why the classifier produced its prediction.
